# FACR + RL Activation Steering
## Factuality-Aware Contrastive Regularization with RL-based Activation Steering

---

## What this notebook does

This notebook investigates hallucination in LLMs from two angles:

**Detection** — finding the critical layer L* where the model's internal representations separate factual from hallucinated responses.

**Correction** — using reinforcement learning to learn a steering vector Δh that pushes hallucinated activations toward the factual region of activation space, potentially correcting the hallucination before it is generated.

---

## The core idea (Experiment F)

When a model is about to hallucinate, its hidden state at layer L* drifts toward a hallucination region in activation space. The RL steering policy learns:

> *If h is in the hallucination region → output Δh such that h + Δh is closer to the factual centroid*

This is not detection. This is intervention — moving the activation to correct the model before it outputs the wrong answer.

---

## Experiments

| Cell | Experiment | What it finds |
|---|---|---|
| 5 | A — Layer discovery | L* per dataset (auto, no hardcoding) |
| 6 | B — Early detection | Earliest reliable token cutoff |
| 7 | C — Transfer | Does geometry generalize across datasets? |
| 8-9 | D+E — Contrastive | Does InfoNCE sharpen the geometry? |
| 10-13 | F — RL Steering | Can RL move hallucinated activations to factual region? |

---

## Before running

1. Replace `your_hf_token_here` in **Cell 2** with your HuggingFace token
2. Runtime → Change runtime type → **A100 GPU**
3. Runtime → **Run all**

> Total runtime: approximately 2-3 hours on A100
> All results saved to `/content/drive/MyDrive/facr_v2/`

### CELL 0 — Setup — Google Drive directories + cached data

Creates Google Drive directories and copies cached data
if it exists from a previous session.
This saves ~36 minutes of data regeneration.

In [ ]:
# ── CELL 0 — Setup ───────────────────────────────────────────────
# Creates Google Drive directories and copies cached data
# if it exists from a previous session.
# This saves ~36 minutes of data regeneration.

import os, shutil

os.makedirs('/content/drive/MyDrive/facr_v2', exist_ok=True)
os.makedirs('/content/drive/MyDrive/facr_v2/step1_results', exist_ok=True)
os.makedirs('/content/drive/MyDrive/facr_v2/step2_results', exist_ok=True)

# copy labeled_data.json if it exists from previous session
src = '/content/drive/MyDrive/facr_checkpoints/labeled_data.json'
dst = '/content/drive/MyDrive/facr_v2/labeled_data.json'
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy(src, dst)
    print("labeled_data.json copied from previous session ✓")
elif os.path.exists(dst):
    print("labeled_data.json already exists ✓")
else:
    print("labeled_data.json not found — will regenerate in Cell 4 (~36 min)")

print("Directories ready ✓")


### CELL 1 — Install all dependencies

Install all required packages.
Run this cell first, then restart runtime if prompted.

In [ ]:
# ── CELL 1 — Install dependencies ────────────────────────────────
# Install all required packages.
# Run this cell first, then restart runtime if prompted.

!pip install transformers datasets peft accelerate bitsandbytes tqdm scikit-learn matplotlib -q


### CELL 2 — Imports + frozen Llama-3.2-3B loading

Loads all libraries and the frozen Llama-3.2-3B-Instruct model.
IMPORTANT: Replace "your_hf_token_here" with your HuggingFace token.
Get one at: https://huggingface.co/settings/tokens
You must have accepted the Llama-3.2 license at:
https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

In [ ]:
# ── CELL 2 — Imports and model loading ───────────────────────────
# Loads all libraries and the frozen Llama-3.2-3B-Instruct model.
#
# IMPORTANT: Replace "your_hf_token_here" with your HuggingFace token.
# Get one at: https://huggingface.co/settings/tokens
# You must have accepted the Llama-3.2 license at:
# https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

import os, json, re, string, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset as TorchDataset

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA

from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    pass  # already mounted

SAVE_DIR  = '/content/drive/MyDrive/facr_v2'
STEP1_DIR = f'{SAVE_DIR}/step1_results'
STEP2_DIR = f'{SAVE_DIR}/step2_results'
os.makedirs(STEP1_DIR, exist_ok=True)
os.makedirs(STEP2_DIR, exist_ok=True)

from huggingface_hub import login
login("your_hf_token_here")  # <-- REPLACE THIS

MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE      = "cuda"
NUM_LAYERS  = 28      # Llama-3.2-3B: 0=embedding, 1-28=transformer blocks
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Device : {DEVICE} | {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
)
model.eval()
print(f"Model loaded ✓ | Layers: {NUM_LAYERS + 1} "
      f"(0=embedding, 1-{NUM_LAYERS}=transformer)")


### CELL 3 — Core framework — auto layer selection, no hardcoded layers

These are the building blocks used by every experiment.
KEY DESIGN PRINCIPLE: No hardcoded layer numbers anywhere.
find_and_evaluate() automatically finds L* — the critical layer
where hallucination geometry peaks — for any model/dataset/cutoff.
THREE-WAY SPLIT (prevents data leakage):
20% → layer selection   (find L* via quick sweep)
60% → probe training    (train logistic probe at L*)
20% → evaluation        (final unbiased AUROC)

In [ ]:
# ── CELL 3 — Core framework functions ────────────────────────────
# These are the building blocks used by every experiment.
#
# KEY DESIGN PRINCIPLE: No hardcoded layer numbers anywhere.
# find_and_evaluate() automatically finds L* — the critical layer
# where hallucination geometry peaks — for any model/dataset/cutoff.
#
# THREE-WAY SPLIT (prevents data leakage):
#   20% → layer selection   (find L* via quick sweep)
#   60% → probe training    (train logistic probe at L*)
#   20% → evaluation        (final unbiased AUROC)

COLORS = {
    'TriviaQA':   'steelblue',
    'HaluEval':   'coral',
    'TruthfulQA': 'mediumseagreen',
    'NQ-Open':    'mediumpurple',
}

# ── Text normalization ────────────────────────────────────────────
def normalize(text):
    """Normalize text for token F1 scoring."""
    text = text.lower().strip()
    text = re.sub(r'\b(1\d{2}|\d{2})0s?\b',
                  lambda m: m.group(0).rstrip('s'), text)
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def token_f1(pred, ref):
    """Token overlap F1 — used to label TriviaQA and NQ-Open responses."""
    p_tok = normalize(pred).split()
    r_tok = normalize(ref).split()
    if not p_tok or not r_tok: return 0.0
    common = set(p_tok) & set(r_tok)
    if not common: return 0.0
    p = len(common) / len(p_tok)
    r = len(common) / len(r_tok)
    return 2 * p * r / (p + r)

def generate_response(question, max_new_tokens=40):
    """Generate a short response from Llama for a given question."""
    prompt = (
        "Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=200
    ).to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.8, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded.split('\n')[0].split('.')[0].strip()

# ── Hidden state extraction ───────────────────────────────────────
def extract_hidden_states(data, layer_idx, cutoff_pct=1.0):
    """
    Extract hidden state at layer_idx for each sample.

    layer_idx  : which transformer layer (0=embedding, 1-28=blocks)
    cutoff_pct : fraction of tokens to use
                 1.0 = full sequence
                 0.2 = first 20% of tokens (early detection)

    Returns:
        hiddens (N, hidden_dim) — float32 cpu tensor
        labels  (N,)            — 1=factual, 0=hallucinated
    """
    hiddens, labels = [], []
    for item in data:
        full_text = item['prompt'] + " " + item['response']
        inputs    = tokenizer(
            full_text, return_tensors='pt',
            truncation=True, max_length=256
        ).to(DEVICE)
        if cutoff_pct < 1.0:
            total  = inputs['input_ids'].shape[1]
            cutoff = max(4, int(total * cutoff_pct))
            inputs = {k: v[:, :cutoff] for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
        h = out.hidden_states[layer_idx][0, -1, :].float().cpu()
        hiddens.append(h)
        labels.append(1 if item['factuality_score'] > 0.5 else 0)
    return torch.stack(hiddens), torch.tensor(labels)

# ── AUROC utilities ───────────────────────────────────────────────
def quick_auroc(hiddens, labels, cv=3):
    """Fast 3-fold AUROC — used only during layer selection sweep."""
    X = hiddens.numpy()
    y = labels.numpy()
    if len(np.unique(y)) < 2: return 0.5
    clf    = LogisticRegression(max_iter=300, C=1.0)
    scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
    return float(scores.mean())

def robust_auroc(hiddens, labels, cv=10):
    """Rigorous 10-fold stratified AUROC — used for final results."""
    X   = hiddens.numpy()
    y   = labels.numpy()
    if len(np.unique(y)) < 2: return 0.5, 0.0
    skf    = StratifiedKFold(
        n_splits=cv, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(X[tr], y[tr])
        proba = clf.predict_proba(X[te])[:, 1]
        scores.append(roc_auc_score(y[te], proba))
    return float(np.mean(scores)), float(np.std(scores))

# ── Core framework: automatic layer selection ─────────────────────
def find_and_evaluate(data, cutoff_pct=1.0, verbose=True):
    """
    THE MAIN FRAMEWORK FUNCTION.
    Automatically finds L* and returns unbiased AUROC.
    No hardcoded layer numbers anywhere in this function.

    HOW IT WORKS:
      1. Split data: 20% selection / 60% training / 20% eval
      2. Sweep all 28 layers on the selection set
      3. Pick layer with highest AUROC (this is L*)
      4. Train logistic probe at L* on training set
      5. Evaluate probe on held-out eval set

    Returns dict with:
      best_layer  : automatically found L*
      layer_depth : L* / NUM_LAYERS (e.g. 0.82 = 82% deep)
      probe_auroc : final unbiased AUROC
      probe       : trained LogisticRegression object
      h_eval      : eval hidden states (for visualization)
      l_eval      : eval labels
    """
    n          = len(data)
    sel_end    = int(0.2 * n)
    train_end  = int(0.8 * n)
    sel_data   = data[:sel_end]
    train_data = data[sel_end:train_end]
    eval_data  = data[train_end:]

    if verbose:
        print(f"  Data split → sel:{len(sel_data)} | "
              f"train:{len(train_data)} | eval:{len(eval_data)}")

    # step 1: sweep all layers
    if verbose:
        print(f"  Sweeping {NUM_LAYERS+1} layers at "
              f"cutoff={int(cutoff_pct*100)}%...")
    layer_aurocs = {}
    for layer_idx in range(NUM_LAYERS + 1):
        h, l = extract_hidden_states(sel_data, layer_idx, cutoff_pct)
        layer_aurocs[layer_idx] = quick_auroc(h, l, cv=3)

    # step 2: pick best layer
    best_layer = max(layer_aurocs, key=layer_aurocs.get)
    if verbose:
        print(f"  L* = layer {best_layer} "
              f"({best_layer/NUM_LAYERS*100:.0f}% depth) | "
              f"sel AUROC = {layer_aurocs[best_layer]:.4f}")

    # step 3: train probe at L*
    h_train, l_train = extract_hidden_states(
        train_data, best_layer, cutoff_pct)
    probe = LogisticRegression(max_iter=1000, C=1.0)
    probe.fit(h_train.numpy(), l_train.numpy())

    # step 4: evaluate on held-out set
    h_eval, l_eval  = extract_hidden_states(
        eval_data, best_layer, cutoff_pct)
    proba       = probe.predict_proba(h_eval.numpy())[:, 1]
    probe_auroc = roc_auc_score(l_eval.numpy(), proba)
    _, probe_std = robust_auroc(h_eval, l_eval, cv=5)

    if verbose:
        print(f"  Final AUROC = {probe_auroc:.4f} ± {probe_std:.4f}")

    return {
        'best_layer':   best_layer,
        'layer_depth':  best_layer / NUM_LAYERS,
        'layer_aurocs': layer_aurocs,
        'probe_auroc':  probe_auroc,
        'probe_std':    probe_std,
        'probe':        probe,
        'h_eval':       h_eval,
        'l_eval':       l_eval,
    }

print("Core framework functions loaded ✓")


### CELL 4 — Dataset loaders — TriviaQA, HaluEval, TruthfulQA, NQ-Open

Loads four datasets used across all experiments.
All datasets are cached to Google Drive after first load.
DATASETS:
TriviaQA   — factual QA, responses generated by model
HaluEval   — pre-labeled hallucinated vs factual QA
TruthfulQA — adversarially constructed questions
NQ-Open    — real Google Natural Questions

In [ ]:
# ── CELL 4 — Dataset loaders ──────────────────────────────────────
# Loads four datasets used across all experiments.
# All datasets are cached to Google Drive after first load.
#
# DATASETS:
#   TriviaQA   — factual QA, responses generated by model
#   HaluEval   — pre-labeled hallucinated vs factual QA
#   TruthfulQA — adversarially constructed questions
#   NQ-Open    — real Google Natural Questions

def load_triviaqa(n=400):
    """
    TriviaQA: generate responses and label with token F1.
    Reuses cached labeled_data.json if available.
    """
    path = f'{SAVE_DIR}/labeled_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        random.seed(RANDOM_SEED)
        random.shuffle(data)
        print(f"TriviaQA  : {len(data[:n])} samples (from cache)")
        return data[:n]

    # generate from scratch if no cache
    print("Generating TriviaQA responses (~36 min)...")
    dataset      = load_dataset(
        "trivia_qa", "rc.nocontext", split="train[:500]")
    labeled_data = []
    for i, example in enumerate(tqdm(dataset)):
        question   = example['question']
        ans        = example['answer']
        prompt     = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        candidates = []
        if ans.get('value'): candidates.append(ans['value'])
        for a in ans.get('aliases', []):
            if isinstance(a, str): candidates.append(a)
        for a in ans.get('normalized_aliases', []):
            if isinstance(a, str): candidates.append(a)
        for _ in range(4):
            resp  = generate_response(question)
            score = max((token_f1(resp, ref)
                         for ref in candidates), default=0.0)
            labeled_data.append({
                'question':         question,
                'response':         resp,
                'factuality_score': float(score),
                'prompt':           prompt
            })
        if (i + 1) % 100 == 0:
            with open(path, 'w') as f:
                json.dump(labeled_data, f)

    with open(path, 'w') as f:
        json.dump(labeled_data, f)
    random.seed(RANDOM_SEED)
    random.shuffle(labeled_data)
    print(f"TriviaQA  : {len(labeled_data[:n])} samples generated")
    return labeled_data[:n]

def load_halueval(n=400):
    """
    HaluEval: pre-labeled QA pairs.
    Field names: question / answer / hallucination (yes/no)
    hallucination=yes → score=0.0 (hallucinated)
    hallucination=no  → score=1.0 (factual)
    """
    path = f'{STEP1_DIR}/halueval_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if len(data) > 0:
            print(f"HaluEval  : {len(data[:n])} samples (from cache)")
            return data[:n]
        os.remove(path)

    ds   = load_dataset("pminervini/HaluEval", "qa_samples", split="data")
    data = []
    for ex in list(ds):
        question     = ex.get('question', '').strip()
        answer       = ex.get('answer', '').strip()
        hallucinated = ex.get('hallucination', 'no').strip().lower()
        if not question or not answer: continue
        prompt = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        data.append({
            'question':         question,
            'response':         answer,
            'factuality_score': 0.0 if hallucinated == 'yes' else 1.0,
            'prompt':           prompt
        })
        if len(data) >= n: break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f:
        json.dump(data[:n], f)
    f_count = sum(1 for d in data[:n] if d['factuality_score'] > 0.5)
    print(f"HaluEval  : {len(data[:n])} samples | "
          f"factual={f_count} | hallu={len(data[:n])-f_count}")
    return data[:n]

def load_truthfulqa(n=400):
    """
    TruthfulQA: adversarially constructed questions.
    best_answer = factual (score=1.0)
    incorrect_answers[0] = hallucinated (score=0.0)
    """
    path = f'{STEP1_DIR}/truthfulqa_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if len(data) > 0:
            print(f"TruthfulQA: {len(data[:n])} samples (from cache)")
            return data[:n]
        os.remove(path)

    ds   = load_dataset("truthful_qa", "generation", split="validation")
    data = []
    for ex in ds:
        question   = ex.get('question', '').strip()
        best_ans   = ex.get('best_answer', '').strip()
        if not best_ans:
            correct  = ex.get('correct_answers', [])
            best_ans = correct[0] if correct else ''
        wrong_list = ex.get('incorrect_answers', [])
        if not question or not best_ans or not wrong_list: continue
        prompt = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        data.append({'question': question, 'response': best_ans,
                     'factuality_score': 1.0, 'prompt': prompt})
        data.append({'question': question, 'response': wrong_list[0],
                     'factuality_score': 0.0, 'prompt': prompt})
        if len(data) >= n: break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f:
        json.dump(data[:n], f)
    print(f"TruthfulQA: {len(data[:n])} samples")
    return data[:n]

def load_nq_open(n=400):
    """
    NQ-Open: real Google search queries.
    Generates responses and labels with token F1.
    """
    path = f'{STEP1_DIR}/nq_labeled.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"NQ-Open   : {len(data[:n])} samples (from cache)")
        return data[:n]

    ds   = load_dataset("nq_open", split="train[:300]")
    data = []
    print("NQ-Open: generating responses (cached after this)...")
    for ex in tqdm(ds):
        question  = ex['question']
        gold_list = ex['answer']
        prompt    = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        for _ in range(2):
            resp  = generate_response(question)
            score = max(token_f1(resp, ref) for ref in gold_list)
            data.append({'question': question, 'response': resp,
                         'factuality_score': float(score),
                         'prompt': prompt})
        if len(data) >= n: break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f:
        json.dump(data[:n], f)
    print(f"NQ-Open   : {len(data[:n])} samples generated and cached")
    return data[:n]

# load all four datasets
print("Loading datasets...")
datasets = {
    'TriviaQA':   load_triviaqa(n=400),
    'HaluEval':   load_halueval(n=400),
    'TruthfulQA': load_truthfulqa(n=400),
    'NQ-Open':    load_nq_open(n=400),
}
print("\nDataset summary:")
for name, data in datasets.items():
    f = sum(1 for d in data if d['factuality_score'] > 0.5)
    h = len(data) - f
    print(f"  {name:<12}: {len(data):>4} samples | "
          f"factual={f} ({100*f/len(data):.0f}%) | "
          f"hallu={h} ({100*h/len(data):.0f}%)")


### CELL 5 — Experiment A — find critical layer L* per dataset

WHAT: Sweep all 28 layers for each dataset and find L* —
the layer where hallucination geometry peaks.
WHY:  Different tasks engage different network depths.
We never hardcode a layer — always find it automatically.
RESULT: L* per dataset, AUROC per dataset at full sequence.

In [ ]:
# ── CELL 5 — Experiment A: critical layer discovery ───────────────
# WHAT: Sweep all 28 layers for each dataset and find L* —
#       the layer where hallucination geometry peaks.
# WHY:  Different tasks engage different network depths.
#       We never hardcode a layer — always find it automatically.
# RESULT: L* per dataset, AUROC per dataset at full sequence.

CUTOFFS = [0.10, 0.20, 0.30, 0.50, 1.00]

print("\n" + "="*60)
print("  EXPERIMENT A — Critical layer discovery")
print("  Finding L* automatically for each dataset")
print("="*60)

full_results = {}
for ds_name, ds_data in datasets.items():
    print(f"\n── {ds_name} " + "─"*38)
    result = find_and_evaluate(ds_data, cutoff_pct=1.0, verbose=True)
    full_results[ds_name] = result

# save results
save_a = {
    k: {'best_layer': v['best_layer'],
        'layer_depth': v['layer_depth'],
        'layer_aurocs': {str(i): a
                         for i, a in v['layer_aurocs'].items()},
        'probe_auroc': v['probe_auroc'],
        'probe_std': v['probe_std']}
    for k, v in full_results.items()
}
with open(f'{STEP1_DIR}/exp_a_results.json', 'w') as f:
    json.dump(save_a, f, indent=2)

print("\n── Experiment A Summary ──────────────────────────────────")
print(f"{'Dataset':<14} {'L*':>4} {'Depth':>8} "
      f"{'AUROC':>8} {'±Std':>8}")
print("-" * 48)
for ds_name, result in full_results.items():
    print(f"{ds_name:<14} {result['best_layer']:>4} "
          f"{result['layer_depth']*100:>7.0f}%  "
          f"{result['probe_auroc']:>8.4f}  "
          f"±{result['probe_std']:.4f}")
depths = [full_results[n]['layer_depth']*100 for n in full_results]
print(f"\n  Mean L* depth: {np.mean(depths):.1f}% ± {np.std(depths):.1f}%")

# set tqa_layer for use in later experiments
tqa_layer = full_results['TriviaQA']['best_layer']
print(f"\n  TriviaQA L* = {tqa_layer} — used in Experiments D-F")


### CELL 6 — Experiment B — early detection sweep (L* per cutoff)

WHAT: Test detection at 10%, 20%, 30%, 50%, 100% token cutoffs.
L* is re-selected independently at each cutoff.
WHY:  We want to know how early in generation we can detect
a hallucination reliably.
RESULT: AUROC curve vs token percentage.

In [ ]:
# ── CELL 6 — Experiment B: early detection sweep ─────────────────
# WHAT: Test detection at 10%, 20%, 30%, 50%, 100% token cutoffs.
#       L* is re-selected independently at each cutoff.
# WHY:  We want to know how early in generation we can detect
#       a hallucination reliably.
# RESULT: AUROC curve vs token percentage.

print("\n" + "="*60)
print("  EXPERIMENT B — Early detection sweep")
print("  L* re-selected independently at each token cutoff")
print("="*60)

early_results = {}
for ds_name in ['TriviaQA', 'HaluEval']:
    early_results[ds_name] = {}
    print(f"\n── {ds_name} " + "─"*38)
    for cutoff in CUTOFFS:
        print(f"\n  Cutoff = {int(cutoff*100)}% tokens:")
        result = find_and_evaluate(
            datasets[ds_name], cutoff_pct=cutoff, verbose=True)
        early_results[ds_name][cutoff] = {
            'best_layer':  result['best_layer'],
            'layer_depth': result['layer_depth'],
            'probe_auroc': result['probe_auroc'],
            'probe_std':   result['probe_std'],
        }

with open(f'{STEP1_DIR}/exp_b_results.json', 'w') as f:
    json.dump(early_results, f, indent=2)

print("\n── Experiment B Summary ──────────────────────────────────")
for ds_name in ['TriviaQA', 'HaluEval']:
    print(f"\n  {ds_name}:")
    print(f"  {'Cutoff':>8} {'L*':>4} {'AUROC':>8} {'±Std':>8}")
    print("  " + "-"*36)
    for cutoff in CUTOFFS:
        r = early_results[ds_name][cutoff]
        print(f"  {int(cutoff*100):>6}%  {r['best_layer']:>4}  "
              f"{r['probe_auroc']:>8.4f}  ±{r['probe_std']:.4f}")


### CELL 7 — Experiment C — cross-dataset transfer

WHAT: Train probe on TriviaQA's L*, test on all other datasets
without retraining.
WHY:  Does the hallucination geometry generalize?
If yes → universal property. If no → task-dependent.
RESULT: Transfer AUROC per dataset.

In [ ]:
# ── CELL 7 — Experiment C: cross-dataset transfer ────────────────
# WHAT: Train probe on TriviaQA's L*, test on all other datasets
#       without retraining.
# WHY:  Does the hallucination geometry generalize?
#       If yes → universal property. If no → task-dependent.
# RESULT: Transfer AUROC per dataset.

print("\n" + "="*60)
print("  EXPERIMENT C — Cross-dataset transfer")
print("  Probe trained on TriviaQA → tested everywhere")
print("="*60)

print(f"\nTriviaQA L* = {tqa_layer} "
      f"({tqa_layer/NUM_LAYERS*100:.0f}% depth)")

# train transfer probe on all TriviaQA data
h_all_tqa, l_all_tqa = extract_hidden_states(
    datasets['TriviaQA'], tqa_layer, 1.0)
transfer_probe = LogisticRegression(max_iter=1000, C=1.0)
transfer_probe.fit(h_all_tqa.numpy(), l_all_tqa.numpy())
print("Transfer probe trained ✓")

transfer_results = {}
print(f"\n{'Dataset':<14} {'Transfer':>10} {'Own L*':>8} "
      f"{'Own AUROC':>10} {'Gap':>8}")
print("-" * 55)
for ds_name, ds_data in datasets.items():
    h, l     = extract_hidden_states(ds_data, tqa_layer, 1.0)
    proba    = transfer_probe.predict_proba(h.numpy())[:, 1]
    t_auroc  = roc_auc_score(l.numpy(), proba)
    own_layer = full_results[ds_name]['best_layer']
    own_auroc = full_results[ds_name]['probe_auroc']
    gap       = own_auroc - t_auroc
    transfer_results[ds_name] = {
        'transfer_auroc': t_auroc, 'own_auroc': own_auroc,
        'own_layer': own_layer, 'gap': gap}
    note = "← train" if ds_name == 'TriviaQA' else "transfer"
    print(f"{ds_name:<14} {t_auroc:>10.4f} {own_layer:>8} "
          f"{own_auroc:>10.4f} {gap:>+8.4f}  {note}")

with open(f'{STEP1_DIR}/exp_c_results.json', 'w') as f:
    json.dump(transfer_results, f, indent=2)

mean_transfer = np.mean([
    v['transfer_auroc']
    for k, v in transfer_results.items() if k != 'TriviaQA'])
print(f"\nMean transfer AUROC (excl. train): {mean_transfer:.4f}")


### CELL 8 — Experiment D — contrastive projection head (InfoNCE)

WHAT: Train a lightweight projection head at L* using InfoNCE
contrastive loss. Trained on mixed cutoffs (20%, 50%, 100%)
so it generalizes to early detection.
WHY:  Can we make the geometry MORE separable than raw hidden states?
RESULT: AUROC before vs after projection per dataset/cutoff.

In [ ]:
# ── CELL 8 — Experiment D: contrastive projection head ───────────
# WHAT: Train a lightweight projection head at L* using InfoNCE
#       contrastive loss. Trained on mixed cutoffs (20%, 50%, 100%)
#       so it generalizes to early detection.
# WHY:  Can we make the geometry MORE separable than raw hidden states?
# RESULT: AUROC before vs after projection per dataset/cutoff.

class ProjectionHead(nn.Module):
    """
    Lightweight MLP that projects hidden states into a shaped space.
    InfoNCE loss pulls same-factuality states together and
    pushes different-factuality states apart.
    Only this head trains — Llama stays frozen.
    """
    def __init__(self, input_dim, proj_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

def infonce_loss(z, factuality_scores, temperature=0.07):
    """
    InfoNCE soft contrastive loss.
    Always positive (no collapse), pulls factual reps together,
    pushes hallucinated reps apart.
    """
    sim     = torch.matmul(z, z.T) / temperature
    f       = factuality_scores.unsqueeze(0) - factuality_scores.unsqueeze(1)
    weights = torch.exp(-f**2 / (2 * 0.5**2))
    weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return -(weights * F.log_softmax(sim, dim=1)).sum(dim=1).mean()

class ContrastiveDataset(TorchDataset):
    def __init__(self, h, s):
        self.h = h
        self.s = s
    def __len__(self): return len(self.h)
    def __getitem__(self, i):
        return {'hidden': self.h[i], 'factuality_score': self.s[i]}

def train_projection_head(train_data, target_layer,
                           proj_dim=256, epochs=10,
                           batch_size=64, lr=1e-3,
                           mixed_cutoffs=None):
    """Train projection head on mixed cutoffs for better early detection."""
    if mixed_cutoffs is None:
        mixed_cutoffs = [0.2, 0.5, 1.0]

    print(f"  Extracting hidden states for cutoffs: "
          f"{[int(c*100) for c in mixed_cutoffs]}%...")
    all_h, all_s = [], []
    for cutoff in mixed_cutoffs:
        h, l = extract_hidden_states(train_data, target_layer, cutoff)
        all_h.append(h)
        all_s.append(l.float())

    h_cat = torch.cat(all_h, dim=0)
    s_cat = torch.cat(all_s, dim=0)

    # balance classes
    f_idx = (s_cat == 1).nonzero().squeeze()
    h_idx = (s_cat == 0).nonzero().squeeze()
    min_n = min(len(f_idx), len(h_idx))
    idx   = torch.cat([f_idx[:min_n], h_idx[:min_n]])
    idx   = idx[torch.randperm(len(idx))]

    ds  = ContrastiveDataset(h_cat[idx], s_cat[idx])
    dl  = torch.utils.data.DataLoader(
        ds, batch_size=batch_size, shuffle=True)
    head = ProjectionHead(h_cat.shape[1], proj_dim).to(DEVICE)
    opt  = torch.optim.Adam(head.parameters(), lr=lr)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    print(f"  Training projection head "
          f"({sum(p.numel() for p in head.parameters()):,} params)...")
    head.train()
    for epoch in range(epochs):
        loss_sum = 0
        for batch in dl:
            h = batch['hidden'].to(DEVICE)
            s = batch['factuality_score'].to(DEVICE)
            opt.zero_grad()
            loss = infonce_loss(head(h), s)
            loss.backward()
            opt.step()
            loss_sum += loss.item()
        if (epoch + 1) % 2 == 0:
            print(f"  Epoch {epoch+1:2d}/{epochs} | "
                  f"loss={loss_sum/len(dl):.4f}")
        sch.step()

    head.eval()
    print("  Projection head trained ✓")
    return head

def eval_with_head(head, data, target_layer, cutoff_pct=1.0):
    h, l = extract_hidden_states(data, target_layer, cutoff_pct)
    with torch.no_grad():
        z = head(h.to(DEVICE)).cpu()
    mean, std = robust_auroc(z, l, cv=10)
    return z, l, mean, std

print("\n" + "="*60)
print("  EXPERIMENT D — Contrastive projection head")
print("  InfoNCE loss at L*, trained on mixed token cutoffs")
print("="*60)

tqa_train = datasets['TriviaQA'][:int(0.8*len(datasets['TriviaQA']))]
print(f"\nTarget L* = {tqa_layer} | Train samples = {len(tqa_train)}")

proj_head = train_projection_head(
    tqa_train, tqa_layer,
    proj_dim=256, epochs=10,
    mixed_cutoffs=[0.2, 0.5, 1.0]
)
torch.save(proj_head.state_dict(),
           f'{STEP2_DIR}/proj_head_layer{tqa_layer}.pt')
print("Projection head saved ✓")


### CELL 9 — Experiment E — before vs after contrastive geometry

WHAT: Compare raw probe AUROC vs projected probe AUROC
at all token cutoffs and all datasets.
WHY:  Does contrastive projection make the geometry more separable?
RESULT: AUROC delta per dataset, PCA visualization.

In [ ]:
# ── CELL 9 — Experiment E: before vs after geometry ──────────────
# WHAT: Compare raw probe AUROC vs projected probe AUROC
#       at all token cutoffs and all datasets.
# WHY:  Does contrastive projection make the geometry more separable?
# RESULT: AUROC delta per dataset, PCA visualization.

print("\n" + "="*60)
print("  EXPERIMENT E — Before vs after contrastive geometry")
print("="*60)

contrastive_results = {}
print("\nEvaluating on all datasets...")
for ds_name, ds_data in datasets.items():
    print(f"\n── {ds_name} ──────────────────────────────────────────")
    ds_eval = ds_data[int(0.8*len(ds_data)):]

    h_raw, l_raw    = extract_hidden_states(ds_eval, tqa_layer, 1.0)
    raw_m, raw_s    = robust_auroc(h_raw, l_raw, cv=10)
    z_proj, l_proj, proj_m, proj_s = eval_with_head(
        proj_head, ds_eval, tqa_layer, 1.0)

    delta = proj_m - raw_m
    contrastive_results[ds_name] = {
        'raw_auroc':  raw_m, 'raw_std':   raw_s,
        'proj_auroc': proj_m, 'proj_std': proj_s,
        'delta': delta, 'h_raw': h_raw,
        'z_proj': z_proj, 'labels': l_raw,
    }
    print(f"  Raw hidden : AUROC = {raw_m:.4f} ± {raw_s:.4f}")
    print(f"  Projected  : AUROC = {proj_m:.4f} ± {proj_s:.4f}")
    print(f"  Delta      : {delta:+.4f}")

# early detection: raw vs projected
print("\n── Early detection raw vs projected (TriviaQA) ──────────")
print(f"{'Cutoff':>8} {'Raw':>10} {'Projected':>12} {'Delta':>8}")
print("-" * 44)

early_contrastive = {}
tqa_eval = datasets['TriviaQA'][int(0.8*len(datasets['TriviaQA'])):]
for cutoff in CUTOFFS:
    h_raw, l_raw    = extract_hidden_states(tqa_eval, tqa_layer, cutoff)
    raw_m, _        = robust_auroc(h_raw, l_raw, cv=5)
    _, _, proj_m, _ = eval_with_head(proj_head, tqa_eval, tqa_layer, cutoff)
    delta = proj_m - raw_m
    early_contrastive[cutoff] = {
        'raw_auroc': raw_m, 'proj_auroc': proj_m, 'delta': delta}
    print(f"{int(cutoff*100):>7}%  {raw_m:>10.4f}  "
          f"{proj_m:>12.4f}  {delta:>+8.4f}")


### CELL 10 — Experiment F — RL activation steering (definitions + SteeringPolicy)
YOUR CORE IDEA:
If we know a hidden state is pointing toward hallucination,
use RL to move that activation in a direction that corrects it.
HOW IT WORKS:
State  : hidden state h at L* (3072-dim vector)
Action : steering vector Δh — which direction to push h
Reward : how much closer h+Δh is to the factual centroid
(positive reward = moved toward factual region)

In [ ]:
# ── CELL 10 — Experiment F: RL activation steering definitions ────
# YOUR CORE IDEA:
#   If we know a hidden state is pointing toward hallucination,
#   use RL to move that activation in a direction that corrects it.
#
# HOW IT WORKS:
#   State  : hidden state h at L* (3072-dim vector)
#   Action : steering vector Δh — which direction to push h
#   Reward : how much closer h+Δh is to the factual centroid
#            (positive reward = moved toward factual region)
#            (negative reward = moved wrong direction)
#
# THE STEERING POLICY:
#   - Detects whether h is in hallucination region (using probe)
#   - If hallucinated: outputs Δh to push toward factual centroid
#   - If factual: outputs small Δh (don't disturb correct states)
#   - Llama stays completely frozen throughout
#
# WHY RL AND NOT SUPERVISED:
#   We don't know the "correct" Δh for each sample.
#   RL discovers the optimal steering direction by trial and error,
#   guided by the reward signal (centroid proximity).

class SteeringPolicy(nn.Module):
    """
    RL policy that learns to steer hallucinated activations
    toward the factual region of activation space.

    Input : hidden state h at L* (3072-dim)
    Output: steering vector Δh (3072-dim, small magnitude)

    The steered state is h + Δh.
    Llama never sees this — we use it for analysis.
    In a real system you would inject h + Δh back into the
    model's forward pass at layer L* to correct generation.

    scale parameter limits how far we can move h.
    Small scale = conservative steering.
    Large scale = aggressive steering.
    """
    def __init__(self, hidden_dim, scale=0.1):
        super().__init__()
        self.scale = scale
        self.net   = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Linear(256, hidden_dim)
        )

    def forward(self, h):
        """Outputs Δh — the steering vector."""
        delta = self.net(h)
        # tanh keeps direction, scale limits magnitude
        # so we can't move h too far from its original position
        return torch.tanh(delta) * self.scale

    def steer(self, h):
        """Returns steered state: h + Δh"""
        return h + self.forward(h)


def compute_steering_reward(h_original, h_steered,
                             factual_centroid, hallu_centroid,
                             true_label):
    """
    Reward for the steering action.

    THIS IS YOUR IDEA IMPLEMENTED:
      Factual state (true_label=1):
        → should NOT be moved (it's already correct)
        → reward = negative of how much we moved it
        → this prevents the policy from disturbing correct states

      Hallucinated state (true_label=0):
        → SHOULD be moved toward factual centroid
        → reward = improvement in similarity to factual centroid
                 + reduction in similarity to hallu centroid
        → positive reward = we successfully steered it toward factual
        → negative reward = we steered it in the wrong direction

    The RL policy learns to maximize this reward,
    which means learning to push hallucinated states
    toward the factual region and leave factual states alone.
    """
    sim_orig_f  = F.cosine_similarity(
        h_original.unsqueeze(0),
        factual_centroid.unsqueeze(0)).item()
    sim_steer_f = F.cosine_similarity(
        h_steered.unsqueeze(0),
        factual_centroid.unsqueeze(0)).item()
    sim_orig_h  = F.cosine_similarity(
        h_original.unsqueeze(0),
        hallu_centroid.unsqueeze(0)).item()
    sim_steer_h = F.cosine_similarity(
        h_steered.unsqueeze(0),
        hallu_centroid.unsqueeze(0)).item()

    if true_label == 0:   # hallucinated — steer toward factual
        # reward = how much closer we got to factual centroid
        # AND how much further we got from hallu centroid
        reward = (sim_steer_f - sim_orig_f) + \
                 (sim_orig_h  - sim_steer_h)
    else:                 # factual — don't disturb
        # penalize unnecessary movement
        movement = F.mse_loss(h_steered, h_original).item()
        reward   = -movement * 10

    return reward, sim_orig_f, sim_steer_f


def train_steering_policy(train_data, target_layer,
                           epochs=30, lr=1e-3,
                           scale=0.1, device='cuda'):
    """
    Train RL activation steering policy.

    Algorithm:
      For each (hidden_state, label) pair:
        1. Compute steering vector Δh via policy
        2. Get steered state h_new = h + Δh
        3. Compute reward based on centroid proximity
        4. Gradient step to maximize reward

      For hallucinated states: maximize movement toward factual centroid
      For factual states: minimize movement (don't disturb)

    This directly implements your idea:
      RL learns to move hallucinated activations
      toward the factual region, potentially correcting
      the model before it outputs the wrong answer.
    """
    print("  Extracting hidden states and computing centroids...")
    h_all, l_all = extract_hidden_states(train_data, target_layer, 1.0)
    h_all        = h_all.to(device)
    l_all        = l_all.to(device)

    f_mask = l_all == 1
    h_mask = l_all == 0

    # factual and hallucinated centroids in activation space
    # these are the targets the RL policy steers toward/away from
    factual_centroid = F.normalize(h_all[f_mask].mean(0), dim=-1)
    hallu_centroid   = F.normalize(h_all[h_mask].mean(0), dim=-1)

    initial_sep = F.cosine_similarity(
        factual_centroid.unsqueeze(0),
        hallu_centroid.unsqueeze(0)).item()

    print(f"  Factual samples     : {f_mask.sum()}")
    print(f"  Hallucinated samples: {h_mask.sum()}")
    print(f"  Initial centroid similarity: {initial_sep:.4f}")
    print(f"  (lower = more separated, -1.0 = perfectly opposite)")

    hidden_dim = h_all.shape[1]
    policy     = SteeringPolicy(hidden_dim, scale=scale).to(device)
    optimizer  = torch.optim.Adam(policy.parameters(), lr=lr)

    total_params = sum(p.numel() for p in policy.parameters())
    print(f"\n  Policy params : {total_params:,}")
    print(f"  Steering scale: {scale} (max perturbation)")
    print(f"  Epochs        : {epochs}")
    print(f"\n  Training...\n")

    best_sep = initial_sep

    for epoch in range(epochs):
        hallu_rews, fact_rews = [], []
        idx = torch.randperm(len(h_all))

        for i in idx:
            h          = h_all[i].unsqueeze(0)
            true_label = l_all[i].item()
            h_steered  = policy.steer(h)

            reward, sim_orig, sim_steer = compute_steering_reward(
                h.squeeze(0), h_steered.squeeze(0),
                factual_centroid, hallu_centroid, true_label)

            if true_label == 0:   # hallucinated: steer toward factual
                # differentiable loss: maximize cosine sim to factual centroid
                loss = -(
                    F.cosine_similarity(
                        F.normalize(h_steered, dim=-1),
                        factual_centroid.unsqueeze(0)) -
                    F.cosine_similarity(
                        F.normalize(h_steered, dim=-1),
                        hallu_centroid.unsqueeze(0))
                ).mean()
                hallu_rews.append(reward)
            else:                  # factual: don't disturb
                # loss: minimize magnitude of steering vector
                delta = policy(h)
                loss  = (delta ** 2).mean() * 10
                fact_rews.append(reward)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()

        # evaluate every 5 epochs
        if (epoch + 1) % 5 == 0:
            policy.eval()
            with torch.no_grad():
                h_new = h_all.clone()
                for i in range(len(h_all)):
                    if l_all[i].item() == 0:   # only steer hallucinated
                        h_new[i] = policy.steer(
                            h_all[i].unsqueeze(0)).squeeze(0)

                new_f_c = F.normalize(h_new[f_mask].mean(0), dim=-1)
                new_h_c = F.normalize(h_new[h_mask].mean(0), dim=-1)
                new_sep = F.cosine_similarity(
                    new_f_c.unsqueeze(0),
                    new_h_c.unsqueeze(0)).item()

                # AUROC after steering
                from sklearn.linear_model import LogisticRegression as LR
                clf   = LR(max_iter=300)
                X_new = h_new.cpu().numpy()
                clf.fit(X_new, l_all.cpu().numpy())
                proba = clf.predict_proba(X_new)[:, 1]
                auroc = roc_auc_score(l_all.cpu().numpy(), proba)

            policy.train()

            print(f"  Epoch {epoch+1:3d}/{epochs} | "
                  f"sep={new_sep:.4f} (init={initial_sep:.4f}) | "
                  f"AUROC={auroc:.4f} | "
                  f"hallu_rew={np.mean(hallu_rews):.3f} | "
                  f"fact_rew={np.mean(fact_rews):.3f}")

            if new_sep < best_sep:
                best_sep = new_sep
                torch.save(policy.state_dict(),
                           f'{STEP2_DIR}/steering_policy_best.pt')

    print(f"\n  Initial separation : {initial_sep:.4f}")
    print(f"  Best separation    : {best_sep:.4f}")
    print(f"  Improvement        : {initial_sep - best_sep:.4f}")
    print(f"  (positive = clusters moved further apart)")

    return policy, factual_centroid, hallu_centroid, initial_sep


print("RL activation steering framework loaded ✓")
print(f"\nYour idea in code:")
print(f"  h at L* → SteeringPolicy → Δh")
print(f"  h + Δh closer to factual centroid → +reward")
print(f"  h + Δh closer to hallu centroid   → -reward")
print(f"  RL learns optimal Δh for each hidden state")


### CELL 11 — Experiment F — train RL steering policy (epochs=30)
Trains the RL activation steering policy on TriviaQA.
Watch the separation value — it should DECREASE over epochs.
Lower separation = clusters are more separated = steering is working.

In [ ]:
# ── CELL 11 — Experiment F: train steering policy ─────────────────
# Trains the RL activation steering policy on TriviaQA.
# Watch the separation value — it should DECREASE over epochs.
# Lower separation = clusters are more separated = steering is working.

print("\n" + "="*60)
print("  EXPERIMENT F — RL Activation Steering Training")
print("  Learning to move hallucinated activations")
print("  toward the factual region at layer L*")
print("="*60)

tqa_train_rl = datasets['TriviaQA'][:int(
    0.8 * len(datasets['TriviaQA']))]

print(f"\nTraining on TriviaQA")
print(f"  L*             : {tqa_layer} "
      f"({tqa_layer/NUM_LAYERS*100:.0f}% depth)")
print(f"  Train samples  : {len(tqa_train_rl)}")
print(f"  Steering scale : 0.1 (conservative)")
print(f"\nWatch: 'sep' should decrease — means clusters moving apart")

steering_policy, f_centroid, h_centroid, initial_sep = \
    train_steering_policy(
        tqa_train_rl,
        target_layer=tqa_layer,
        epochs=30,
        lr=1e-3,
        scale=0.1,
        device=DEVICE
    )


### CELL 12 — Experiment F — evaluate steering on TriviaQA eval set
WHAT: Compare hidden states before and after steering.
Metrics:
AUROC          : can a probe separate factual from hallucinated?
(should INCREASE after steering)
Centroid sim   : cosine similarity between factual/hallu centroids
(should DECREASE — lower = more separated)
Factual score  : mean distance of factual states from hallu centroid
(should be NEGATIVE = factual states are far from hallu)

In [ ]:
# ── CELL 12 — Experiment F: evaluate steering ────────────────────
# WHAT: Compare hidden states before and after steering.
# Metrics:
#   AUROC          : can a probe separate factual from hallucinated?
#                    (should INCREASE after steering)
#   Centroid sim   : cosine similarity between factual/hallu centroids
#                    (should DECREASE — lower = more separated)
#   Factual score  : mean distance of factual states from hallu centroid
#                    (should be NEGATIVE = factual states are far from hallu)
#   Hallu score    : mean distance of hallu states from factual centroid
#                    (should be POSITIVE = hallu states are far from factual)

def evaluate_steering(policy, eval_data, target_layer,
                       factual_centroid, hallu_centroid,
                       probe, device='cuda'):
    """
    Evaluate steering policy.

    For each sample:
      1. Extract h at L*
      2. If probe flags as hallucinated → apply steering (h + Δh)
      3. If probe flags as factual → leave h unchanged
      4. Measure: did steering improve separability?

    This simulates the real use case:
      - Probe detects hallucination in real time
      - Steering policy corrects the activation
      - Model would generate more factually from corrected state
    """
    policy.eval()
    h_eval, l_eval = extract_hidden_states(eval_data, target_layer, 1.0)
    h_eval_dev     = h_eval.to(device)
    labels         = l_eval.numpy()

    # apply steering only to probe-flagged samples
    h_steered = h_eval_dev.clone()
    n_steered = 0
    with torch.no_grad():
        for i in range(len(h_eval_dev)):
            h_i   = h_eval_dev[i].unsqueeze(0)
            score = probe.predict_proba(h_i.cpu().numpy())[0, 1]
            if score > 0.5:   # probe says hallucinated → steer it
                h_steered[i] = policy.steer(h_i).squeeze(0)
                n_steered    += 1

    h_steered_cpu = h_steered.cpu()

    # AUROC before and after
    from sklearn.model_selection import cross_val_score as cvs
    from sklearn.linear_model import LogisticRegression as LR
    clf = LR(max_iter=1000)

    a_before = cvs(clf, h_eval.numpy(), labels,
                   cv=5, scoring='roc_auc').mean()
    a_after  = cvs(clf, h_steered_cpu.numpy(), labels,
                   cv=5, scoring='roc_auc').mean()

    # centroid separation before and after
    fm = l_eval == 1
    hm = l_eval == 0

    fc_b = F.normalize(h_eval[fm].mean(0), dim=-1)
    hc_b = F.normalize(h_eval[hm].mean(0), dim=-1)
    sep_b = F.cosine_similarity(
        fc_b.unsqueeze(0), hc_b.unsqueeze(0)).item()

    fc_a = F.normalize(h_steered_cpu[fm].mean(0), dim=-1)
    hc_a = F.normalize(h_steered_cpu[hm].mean(0), dim=-1)
    sep_a = F.cosine_similarity(
        fc_a.unsqueeze(0), hc_a.unsqueeze(0)).item()

    # mean hallucination scores (sim_hallu - sim_factual per sample)
    scores_before, scores_after = [], []
    for i in range(len(h_eval)):
        sb = (F.cosine_similarity(h_eval[i:i+1],
                                   hc_b.unsqueeze(0)).item() -
              F.cosine_similarity(h_eval[i:i+1],
                                   fc_b.unsqueeze(0)).item())
        sa = (F.cosine_similarity(h_steered_cpu[i:i+1],
                                   hc_a.unsqueeze(0)).item() -
              F.cosine_similarity(h_steered_cpu[i:i+1],
                                   fc_a.unsqueeze(0)).item())
        scores_before.append(sb)
        scores_after.append(sa)

    sb = np.array(scores_before)
    sa = np.array(scores_after)

    return {
        'auroc_before':      a_before,
        'auroc_after':       a_after,
        'auroc_delta':       a_after - a_before,
        'sep_before':        sep_b,
        'sep_after':         sep_a,
        'sep_delta':         sep_a - sep_b,
        'mean_hallu_before': sb[labels == 0].mean(),
        'mean_hallu_after':  sa[labels == 0].mean(),
        'mean_fact_before':  sb[labels == 1].mean(),
        'mean_fact_after':   sa[labels == 1].mean(),
        'n_steered':         n_steered,
        'n_total':           len(eval_data),
    }

tqa_eval_rl  = datasets['TriviaQA'][int(
    0.8*len(datasets['TriviaQA'])):]
tqa_probe    = full_results['TriviaQA']['probe']

print("\nEvaluating steering on TriviaQA eval set...")
steer_results = evaluate_steering(
    steering_policy, tqa_eval_rl, tqa_layer,
    f_centroid, h_centroid, tqa_probe, device=DEVICE)

print(f"\n── Steering Evaluation Results ────────────────────────")
print(f"\n  {'Metric':<35} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*62}")
print(f"  {'Probe AUROC':<35} "
      f"{steer_results['auroc_before']:>8.4f} "
      f"{steer_results['auroc_after']:>8.4f} "
      f"{steer_results['auroc_delta']:>+8.4f}")
print(f"  {'Centroid cosine similarity':<35} "
      f"{steer_results['sep_before']:>8.4f} "
      f"{steer_results['sep_after']:>8.4f} "
      f"{steer_results['sep_delta']:>+8.4f}")
print(f"  {'  (lower = more separated)':<35}")
print(f"  {'Mean hallu score (hallu samples)':<35} "
      f"{steer_results['mean_hallu_before']:>+8.4f} "
      f"{steer_results['mean_hallu_after']:>+8.4f} "
      f"{steer_results['mean_hallu_after']-steer_results['mean_hallu_before']:>+8.4f}")
print(f"  {'Mean hallu score (factual samples)':<35} "
      f"{steer_results['mean_fact_before']:>+8.4f} "
      f"{steer_results['mean_fact_after']:>+8.4f} "
      f"{steer_results['mean_fact_after']-steer_results['mean_fact_before']:>+8.4f}")
print(f"\n  Samples steered: {steer_results['n_steered']} / "
      f"{steer_results['n_total']}")

print(f"\n── Interpretation ────────────────────────────────────")
if steer_results['sep_delta'] < 0:
    print(f"  ✓ Centroid separation improved — "
          f"clusters moved further apart")
    print(f"    Hallucinated activations successfully steered "
          f"toward factual region")
else:
    print(f"  ~ Centroid separation did not improve")
    print(f"    Try: larger scale, more epochs, or different lr")

if steer_results['auroc_delta'] > 0:
    print(f"  ✓ AUROC improved — steered states are more separable")
else:
    print(f"  ~ AUROC did not improve after steering")


### CELL 13 — Experiment F — cross-dataset steering evaluation
WHAT: Test steering policy (trained on TriviaQA) on all 4 datasets.
WHY:  Does the steering direction generalize?
RESULT: AUROC and separation before/after per dataset.

In [ ]:
# ── CELL 13 — Experiment F: cross-dataset steering ───────────────
# WHAT: Test steering policy (trained on TriviaQA) on all 4 datasets.
# WHY:  Does the steering direction generalize?
# RESULT: AUROC and separation before/after per dataset.

print(f"\n── Cross-dataset steering evaluation ─────────────────")
print(f"  Policy trained on TriviaQA L*={tqa_layer}\n")
print(f"  {'Dataset':<14} {'AUROC bef':>10} {'AUROC aft':>10} "
      f"{'Sep bef':>8} {'Sep aft':>8} {'Steered':>8}")
print(f"  {'-'*62}")

cross_steer = {}
for ds_name, ds_data in datasets.items():
    ds_eval  = ds_data[int(0.8*len(ds_data)):]
    ds_probe = full_results[ds_name]['probe']

    res = evaluate_steering(
        steering_policy, ds_eval, tqa_layer,
        f_centroid, h_centroid, ds_probe, device=DEVICE)
    cross_steer[ds_name] = res

    print(f"  {ds_name:<14} {res['auroc_before']:>10.4f} "
          f"{res['auroc_after']:>10.4f} "
          f"{res['sep_before']:>8.4f} "
          f"{res['sep_after']:>8.4f} "
          f"{res['n_steered']:>7}/{res['n_total']}")

with open(f'{STEP2_DIR}/exp_f_steering_results.json', 'w') as f:
    json.dump(
        {k: {kk: float(vv) if isinstance(vv, (float, np.floating))
             else int(vv)
             for kk, vv in v.items()
             if not isinstance(vv, np.ndarray)}
         for k, v in cross_steer.items()},
        f, indent=2)

print(f"\n── What steering learned ─────────────────────────────")
print(f"  For each hallucinated hidden state at layer {tqa_layer}:")
print(f"  → RL policy outputs Δh (steering vector)")
print(f"  → h + Δh is closer to the factual centroid")
print(f"  → In a deployed system, injecting h + Δh into")
print(f"    Llama at layer {tqa_layer} would steer generation")
print(f"    away from the hallucinated answer")


### CELL 14 — All figures — 6 publication-ready plots saved to Drive
Generates 6 publication-ready figures saved to Google Drive.
figure 1: layer AUROC curves

In [ ]:
# ── CELL 14 — All figures ─────────────────────────────────────────
# Generates 6 publication-ready figures saved to Google Drive.

# figure 1: layer AUROC curves
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for ax, (ds_name, result) in zip(axes, full_results.items()):
    color  = COLORS[ds_name]
    layers = sorted(result['layer_aurocs'].keys())
    aurocs = [result['layer_aurocs'][l] for l in layers]
    best   = result['best_layer']
    ax.plot(layers, aurocs, '-', color=color, linewidth=1.5, alpha=0.8)
    ax.scatter(layers, aurocs, color=color, s=20, zorder=4)
    ax.axvline(best, color='black', linestyle='--', linewidth=1.2)
    ax.scatter([best], [result['layer_aurocs'][best]],
               color='black', s=100, zorder=5,
               label=f"L*={best} ({best/NUM_LAYERS*100:.0f}%)")
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(f"{ds_name}\nAUROC={result['probe_auroc']:.3f}",
                 fontsize=10)
    ax.set_xlabel('Layer', fontsize=9)
    ax.set_ylim(0.4, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    if ax == axes[0]:
        ax.set_ylabel('AUROC (probe)', fontsize=10)
plt.suptitle('Figure 1: Auto L* discovery — AUROC by layer across datasets',
             fontsize=11)
plt.tight_layout()
plt.savefig(f'{STEP1_DIR}/fig1_layer_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved ✓")

# figure 2: early detection
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ds_name in zip(axes, ['TriviaQA', 'HaluEval']):
    cutoffs    = sorted(early_results[ds_name].keys())
    auroc_vals = [early_results[ds_name][c]['probe_auroc'] for c in cutoffs]
    layers     = [early_results[ds_name][c]['best_layer']  for c in cutoffs]
    xi         = range(len(cutoffs))
    ax.plot(xi, auroc_vals, 'o-', color=COLORS[ds_name], linewidth=2, markersize=8)
    for i, (a, l) in enumerate(zip(auroc_vals, layers)):
        ax.annotate(f"L*={l}", (i, a), textcoords="offset points",
                    xytext=(0, 10), ha='center', fontsize=8)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
    ax.axhline(0.7, color='green', linestyle='--', alpha=0.5, label='0.7 threshold')
    ax.set_xticks(xi)
    ax.set_xticklabels([f"{int(c*100)}%" for c in cutoffs], fontsize=10)
    ax.set_xlabel('Tokens generated', fontsize=10)
    ax.set_ylabel('AUROC', fontsize=10)
    ax.set_title(f'Figure 2: {ds_name} — early detection', fontsize=10)
    ax.set_ylim(0.4, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(f'{STEP1_DIR}/fig2_early_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved ✓")

# figure 3: PCA before vs after contrastive projection
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
res = contrastive_results['TriviaQA']
for ax, hiddens, title in [
    (axes[0], res['h_raw'],  f'Figure 3a: Raw (layer {tqa_layer})'),
    (axes[1], res['z_proj'], 'Figure 3b: After contrastive projection')
]:
    y       = res['labels'].numpy()
    reduced = PCA(n_components=2).fit_transform(hiddens.numpy())
    fm      = y == 1
    ax.scatter(reduced[fm,  0], reduced[fm,  1],
               c='steelblue', alpha=0.6, s=18, label='Factual')
    ax.scatter(reduced[~fm, 0], reduced[~fm, 1],
               c='coral',     alpha=0.6, s=18, label='Hallucinated')
    h_t = torch.tensor(hiddens.numpy())
    sep = F.cosine_similarity(
        h_t[fm].mean(0).unsqueeze(0),
        h_t[~fm].mean(0).unsqueeze(0)).item()
    ax.set_title(f'{title}\nCentroid sim={sep:.3f}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig3_pca_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved ✓")

# figure 4: early detection raw vs projected
fig, ax = plt.subplots(figsize=(8, 5))
cutoffs   = sorted(early_contrastive.keys())
raw_vals  = [early_contrastive[c]['raw_auroc']  for c in cutoffs]
proj_vals = [early_contrastive[c]['proj_auroc'] for c in cutoffs]
xi        = range(len(cutoffs))
ax.plot(xi, raw_vals,  'o-', color='steelblue', linewidth=2,
        markersize=8, label='Raw hidden states')
ax.plot(xi, proj_vals, 's-', color='coral', linewidth=2,
        markersize=8, label='Contrastive projection')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0.7, color='green', linestyle='--', alpha=0.5)
ax.set_xticks(xi)
ax.set_xticklabels([f"{int(c*100)}%" for c in cutoffs], fontsize=11)
ax.set_xlabel('Tokens generated', fontsize=11)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_title(f'Figure 4: Early detection — raw vs projected (L*={tqa_layer})')
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig4_early_contrastive.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 4 saved ✓")

# figure 5: RL steering before vs after PCA
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
tqa_eval_plot = datasets['TriviaQA'][int(0.8*len(datasets['TriviaQA'])):]
h_plot, l_plot = extract_hidden_states(tqa_eval_plot, tqa_layer, 1.0)
h_plot_dev     = h_plot.to(DEVICE)
h_steered_plot = h_plot_dev.clone()
steering_policy.eval()
with torch.no_grad():
    for i in range(len(h_plot_dev)):
        h_i   = h_plot_dev[i].unsqueeze(0)
        score = tqa_probe.predict_proba(h_i.cpu().numpy())[0, 1]
        if score > 0.5:
            h_steered_plot[i] = steering_policy.steer(h_i).squeeze(0)
h_steered_plot_cpu = h_steered_plot.cpu()
y_plot = l_plot.numpy()

for ax, hiddens, title in [
    (axes[0], h_plot,              f'Figure 5a: Before steering (L*={tqa_layer})'),
    (axes[1], h_steered_plot_cpu,  'Figure 5b: After RL activation steering')
]:
    reduced = PCA(n_components=2).fit_transform(hiddens.numpy())
    fm      = y_plot == 1
    ax.scatter(reduced[fm,  0], reduced[fm,  1],
               c='steelblue', alpha=0.6, s=18, label='Factual')
    ax.scatter(reduced[~fm, 0], reduced[~fm, 1],
               c='coral',     alpha=0.6, s=18, label='Hallucinated')
    h_t = torch.tensor(hiddens.numpy())
    sep = F.cosine_similarity(
        h_t[fm].mean(0).unsqueeze(0),
        h_t[~fm].mean(0).unsqueeze(0)).item()
    ax.set_title(f'{title}\nCentroid sim={sep:.3f}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.suptitle('RL activation steering: before vs after', fontsize=12)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig5_rl_steering_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 5 saved ✓")

# figure 6: cross-dataset summary
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ds_list = list(cross_steer.keys())
x       = np.arange(len(ds_list))
w       = 0.35

ax = axes[0]
before_aurocs = [cross_steer[d]['auroc_before'] for d in ds_list]
after_aurocs  = [cross_steer[d]['auroc_after']  for d in ds_list]
ax.bar(x-w/2, before_aurocs, w, label='Before steering',
       color='steelblue', alpha=0.8)
ax.bar(x+w/2, after_aurocs,  w, label='After steering',
       color='coral',     alpha=0.8)
for i, (b, a) in enumerate(zip(before_aurocs, after_aurocs)):
    ax.text(i-w/2, b+0.005, f'{b:.3f}', ha='center', va='bottom', fontsize=7)
    ax.text(i+w/2, a+0.005, f'{a:.3f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(ds_list, fontsize=9)
ax.set_ylabel('AUROC'); ax.set_ylim(0.4, 1.1)
ax.set_title('Figure 6a: AUROC before/after steering'); ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

ax = axes[1]
before_seps = [cross_steer[d]['sep_before'] for d in ds_list]
after_seps  = [cross_steer[d]['sep_after']  for d in ds_list]
ax.bar(x-w/2, before_seps, w, label='Before steering',
       color='steelblue', alpha=0.8)
ax.bar(x+w/2, after_seps,  w, label='After steering',
       color='coral',     alpha=0.8)
for i, (b, a) in enumerate(zip(before_seps, after_seps)):
    ax.text(i-w/2, b+0.005, f'{b:.3f}', ha='center', va='bottom', fontsize=7)
    ax.text(i+w/2, a+0.005, f'{a:.3f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(ds_list, fontsize=9)
ax.set_ylabel('Centroid cosine similarity (lower = better)')
ax.set_title('Figure 6b: Centroid separation before/after')
ax.legend(fontsize=9); ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig6_cross_dataset_steering.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 6 saved ✓")


### CELL 15 — Final results table — all experiments, paper ready
Complete paper-ready summary of all experiments.

In [ ]:
# ── CELL 15 — Final results table ────────────────────────────────
# Complete paper-ready summary of all experiments.

print("\n" + "="*60)
print("  FINAL RESULTS TABLE — PAPER READY")
print("="*60)
print(f"\nModel  : Llama-3.2-3B-Instruct (frozen throughout)")
print(f"Method : Auto L* + contrastive projection + RL steering")

print(f"\n── Table 1: Critical layer discovery (Exp A) ─────────")
print(f"{'Dataset':<14} {'L*':>4} {'Depth':>8} "
      f"{'AUROC':>8} {'±Std':>8}")
print("-" * 50)
for ds_name in datasets.keys():
    r = full_results[ds_name]
    print(f"{ds_name:<14} {r['best_layer']:>4} "
          f"{r['layer_depth']*100:>7.0f}%  "
          f"{r['probe_auroc']:>8.4f}  ±{r['probe_std']:.4f}")
depths_list = [full_results[n]['layer_depth']*100 for n in datasets]
print(f"\n  Mean L* depth: {np.mean(depths_list):.1f}% "
      f"± {np.std(depths_list):.1f}%")

print(f"\n── Table 2: Early detection at L*=13 (Exp B) ─────────")
print(f"{'Cutoff':>8} {'Raw AUROC':>12} "
      f"{'Proj AUROC':>12} {'Delta':>8}")
print("-" * 46)
for cutoff in CUTOFFS:
    r = early_contrastive[cutoff]
    print(f"{int(cutoff*100):>7}%  {r['raw_auroc']:>12.4f}  "
          f"{r['proj_auroc']:>12.4f}  {r['delta']:>+8.4f}")

print(f"\n── Table 3: Cross-dataset transfer (Exp C) ───────────")
print(f"{'Dataset':<14} {'Own L*':>6} {'Own AUROC':>10} "
      f"{'Transfer':>10} {'Gap':>8}")
print("-" * 52)
for ds_name in datasets.keys():
    t = transfer_results[ds_name]
    print(f"{ds_name:<14} {t['own_layer']:>6}  "
          f"{t['own_auroc']:>10.4f}  "
          f"{t['transfer_auroc']:>10.4f}  {t['gap']:>+8.4f}")

print(f"\n── Table 4: Contrastive projection (Exp D+E) ─────────")
print(f"{'Dataset':<14} {'Raw':>8} {'Projected':>10} {'Delta':>8}")
print("-" * 44)
for ds_name, r in contrastive_results.items():
    print(f"{ds_name:<14} {r['raw_auroc']:>8.4f}  "
          f"{r['proj_auroc']:>10.4f}  {r['delta']:>+8.4f}")

print(f"\n── Table 5: RL activation steering (Exp F) ───────────")
print(f"{'Dataset':<14} {'AUROC bef':>10} {'AUROC aft':>10} "
      f"{'Sep bef':>8} {'Sep aft':>8} {'AUROC Δ':>8}")
print("-" * 62)
for ds_name, res in cross_steer.items():
    print(f"{ds_name:<14} {res['auroc_before']:>10.4f} "
          f"{res['auroc_after']:>10.4f} "
          f"{res['sep_before']:>8.4f} "
          f"{res['sep_after']:>8.4f} "
          f"{res['auroc_delta']:>+8.4f}")

print(f"\n── Key findings ──────────────────────────────────────")
all_layers = [full_results[n]['best_layer'] for n in datasets]
print(f"1. L* found automatically: {all_layers}")
print(f"   Mean depth: {np.mean(depths_list):.1f}% ± "
      f"{np.std(depths_list):.1f}% (task-dependent)")
xfer_vals = [v['transfer_auroc']
             for k, v in transfer_results.items() if k != 'TriviaQA']
print(f"2. Transfer AUROC mean: {np.mean(xfer_vals):.4f}")
c_deltas = [r['delta'] for r in contrastive_results.values()]
print(f"3. Contrastive projection mean delta: "
      f"{np.mean(c_deltas):+.4f}")
steer_deltas = [v['auroc_delta'] for v in cross_steer.values()]
print(f"4. RL steering AUROC delta: "
      f"{np.mean(steer_deltas):+.4f}")
steer_sep = [v['sep_delta'] for v in cross_steer.values()]
print(f"5. RL steering sep delta: "
      f"{np.mean(steer_sep):+.4f} "
      f"(negative = better separation)")

print(f"\nAll figures saved:")
for fig_file in [
    f'{STEP1_DIR}/fig1_layer_curves.png',
    f'{STEP1_DIR}/fig2_early_detection.png',
    f'{STEP2_DIR}/fig3_pca_before_after.png',
    f'{STEP2_DIR}/fig4_early_contrastive.png',
    f'{STEP2_DIR}/fig5_rl_steering_pca.png',
    f'{STEP2_DIR}/fig6_cross_dataset_steering.png',
]:
    print(f"  {fig_file}")
print("="*60)
